In [2]:
# 读取数据
import pandas as pd
trip = pd.read_csv('../../../raw_data/04_datacamp/cleaning_data_in_python/ch1_common_data_problems/trip.csv')


In [3]:
# 查看数据详细信息
print(trip.info())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 669959 entries, 0 to 669958
Data columns (total 11 columns):
 #   Column              Non-Null Count   Dtype 
---  ------              --------------   ----- 
 0   id                  669959 non-null  int64 
 1   duration            669959 non-null  int64 
 2   start_date          669959 non-null  object
 3   start_station_name  669959 non-null  object
 4   start_station_id    669959 non-null  int64 
 5   end_date            669959 non-null  object
 6   end_station_name    669959 non-null  object
 7   end_station_id      669959 non-null  int64 
 8   bike_id             669959 non-null  int64 
 9   subscription_type   669959 non-null  object
 10  zip_code            663340 non-null  object
dtypes: int64(5), object(6)
memory usage: 56.2+ MB
None


# 数据清洗与类型转换分析报告

## 1. 数据概览 (Data Overview)
根据 `df.info()` 显示，该数据集共包含 **669,959** 条记录和 **11** 个字段。目前内存占用约为 **56.2+ MB**。

## 2. 核心字段分析与预处理建议

### 2.1 标识符列 (ID Columns)
* **涉及字段**：`id`, `bike_id`, `start_station_id`, `end_station_id`
* **现状**：当前均为 `int64`（64位整数）类型，证明这几列数据中并不存在缺失值以及小数点。
* **分析**：
    * **非数值属性**：ID 本质上是标签（Labels），其数值大小无实际物理意义。
    * **风险规避**：保留为数值型可能导致在自动生成的统计报告中出现无效的数学运算，如求 ID 的平均值。
    * **建议**：统一转换为 **`str` (Object)** 类型，以确保逻辑严谨，并增强后续在多表关联时的稳定性。

### 2.2 时间序列列 (DateTime Columns)
* **涉及字段**：`start_date`, `end_date`
* **现状**：当前为 `object`（通常为字符串）类型。
* **分析**：
    * **脏数据风险**：`object` 类型往往暗示数据中混入了无法被 Pandas 自动解析的非日期字符。
    * **分析受限**：目前无法直接进行骑行时长计算或提取周、小时等时间维度进行趋势分析。
    * **建议**：转换为 **`datetime64`** 类型。建议使用 `errors='coerce'` 策略，将无法解析的脏数据识别为 `NaT`（空时间值）以便后续集中清理。

### 2.3 分类变量优化 (Categorical Columns)
* **涉及字段**：`subscription_type`
* **现状**：当前为 `object` 类型。
* **分析**：
    * **低基数特征**：该字段通常仅包含几种固定的分类取值，如 Subscriber 或 Customer。
    * **极值判断**：通过`describe`以及`sort_values()`判断该字段是否存在不符合常识的数据。
    * **性能提升**：转换为 **`category`** 类型可大幅压缩内存消耗（预计降低 80% 以上），并显著加速后续的分组聚合操作。

### 2.4 缺失值与地理信息 (Geographic Data)
* **涉及字段**：`zip_code`
* **现状**：存在约 **6,619** 条缺失值（占比约 1%）。
* **分析**：
    * **处理策略**：缺失比例较低，建议填充为 `'Unknown'` 保持数据完整性。
    * **类型安全**：必须确保为 **字符串** 格式。若误转为数值型，邮政编码开头的“0”会丢失（例如 02138 会错误变成 2138），导致地理信息失真。

## 3. 进阶优化建议 (Advanced Suggestions)

| 字段名 | 建议操作 | 分析理由 |
| :--- | :--- | :--- |
| **duration** | 降级为 `int32` | 骑行时长通常不会超过 $2^{31}-1$ 秒，降级可节省内存占用。 |
| **station_name** | 转换为 `category` | 站点名称高度重复，使用分类类型优于纯字符串存储。 |
| **一致性检查** | ID-Name 映射分析 | 应核查 `start_station_id` 与名称是否存在“一对多”的映射错误。 |

## 4. 结论
当前数据集虽然结构完整，但存在 **类型不匹配** 和 **内存冗余** 问题。通过上述“标签化、时间化、分类化”的预处理建议，不仅能提高后续探索性数据分析（EDA）的准确性，还能为大规模数据运算提供更高的效率。

In [4]:
import pandas as pd

pd.set_option('display.max_columns', None) 

pd.set_option('display.expand_frame_repr', False)





In [5]:
# 查看数据前几行
print(trip.head())

     id  duration       start_date        start_station_name  start_station_id         end_date          end_station_name  end_station_id  bike_id subscription_type zip_code
0  4576        63  8/29/2013 14:13  South Van Ness at Market                66  8/29/2013 14:14  South Van Ness at Market              66      520        Subscriber    94127
1  4607        70  8/29/2013 14:42        San Jose City Hall                10  8/29/2013 14:43        San Jose City Hall              10      661        Subscriber    95138
2  4130        71  8/29/2013 10:16   Mountain View City Hall                27  8/29/2013 10:17   Mountain View City Hall              27       48        Subscriber    97214
3  4251        77  8/29/2013 11:29        San Jose City Hall                10  8/29/2013 11:30        San Jose City Hall              10       26        Subscriber    95060
4  4299        83  8/29/2013 12:02  South Van Ness at Market                66  8/29/2013 12:04            Market at 10th         

In [6]:
# 查看 'subscription_type' 列的分类字段数量
print(trip['subscription_type'].value_counts())

subscription_type
Subscriber    566746
Customer      103213
Name: count, dtype: int64


# 打印结果分析
这里只出现了两个类别：`Subscriber` 和 `Customer` ,但并不代表数据很干净，如果数据全都是 `Subscriber ` 和 `Customer`（带空格），`value_counts` 依然只会显示一行。
# 预防措施
现在的数据集可能干净，但如果明天加载了一个新的月份的数据，那个数据里万一有`空格`呢？可以利用 `strip()` 去掉`空格`保证脚本在任何时候都能跑通。

In [7]:
# 去掉 'subscription_type' 中首尾可能隐藏的空格
trip['subscription_type'] = trip['subscription_type'].str.strip()

In [8]:
# 将 'id' 列转换为 'str' 类型
id_columns = ['id', 'bike_id', 'start_station_id', 'end_station_id']
trip[id_columns] = trip[id_columns].astype(str)
print(trip.info())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 669959 entries, 0 to 669958
Data columns (total 11 columns):
 #   Column              Non-Null Count   Dtype 
---  ------              --------------   ----- 
 0   id                  669959 non-null  object
 1   duration            669959 non-null  int64 
 2   start_date          669959 non-null  object
 3   start_station_name  669959 non-null  object
 4   start_station_id    669959 non-null  object
 5   end_date            669959 non-null  object
 6   end_station_name    669959 non-null  object
 7   end_station_id      669959 non-null  object
 8   bike_id             669959 non-null  object
 9   subscription_type   669959 non-null  object
 10  zip_code            663340 non-null  object
dtypes: int64(1), object(10)
memory usage: 56.2+ MB
None


In [9]:
# 将start_date和end_date的类型转换为datetime64
trip['start_date'] = pd.to_datetime(trip['start_date'],errors='coerce')
trip['end_date'] = pd.to_datetime(trip['end_date'],errors= 'coerce')

print(trip[['start_date','end_date']].info())

# 检查转换后产生了多少缺失时间
print(f"start_date无法解析的脏数据数量：{trip['start_date'].isna().sum()}")
print(f"end_date无法解析的脏数据数量：{trip['end_date'].isna().sum()}")

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 669959 entries, 0 to 669958
Data columns (total 2 columns):
 #   Column      Non-Null Count   Dtype         
---  ------      --------------   -----         
 0   start_date  669959 non-null  datetime64[ns]
 1   end_date    669959 non-null  datetime64[ns]
dtypes: datetime64[ns](2)
memory usage: 10.2 MB
None
start_date无法解析的脏数据数量：0
end_date无法解析的脏数据数量：0


In [10]:
# 查看 'duration' 列的数值边界（换算成小时）
trip['duration_hours'] = trip['duration'] / 3600
print(trip['duration_hours'].describe())

count    669959.000000
mean          0.307764
std           6.182066
min           0.016667
25%           0.095556
50%           0.143611
75%           0.209722
max        4797.333333
Name: duration_hours, dtype: float64


# 打印结果分析
最大时长为 `4797.333333` ,这不符合常识，是否还存在比这个`异常值` 更小，但依然不符合逻辑的 `时长` ？因此，我决定采取`降序排列的方法`，查看前50个 `最大时长`。

In [11]:
# 排序
print(trip['duration_hours'].sort_values(ascending=False).head(50))

573566    4797.333333
382718     593.611111
440339     514.608333
371066     314.872222
80510      200.621111
606063     200.126111
223016     199.022222
195379     198.705278
421839     191.360833
524521     182.205278
287337     179.103056
93400      172.033889
443792     169.788889
524518     167.316111
20535      165.976944
418295     165.152778
119830     162.876667
152689     155.775556
393058     153.526944
505391     147.566667
252081     143.848889
517402     140.411944
379557     139.615833
640982     135.954167
155325     129.153333
619315     126.975000
43549      119.273333
287336     114.401667
574968     113.932500
531427     112.575278
574943     112.361944
574966     112.176944
568195     105.267778
618556      97.802778
491236      97.473056
192979      96.919444
391004      96.725278
341115      96.262500
232372      91.587500
24068       91.515556
657384      90.215278
558416      88.615833
192978      88.398889
12725       85.966389
579197      84.828333
108144    

# 打印结果分析
    前50的骑行时长最低为80小时，依然不符合常识，因此我决定设置一个阈值，将超出阈值的数据剔除。

In [12]:
# 设置阈值
threshold = 24

# 筛选出小于阈值的数据
trip_clean = trip[trip['duration_hours'] < threshold].copy()

# 打印前后数据长度，并计算剔除的数据量
print(f"原始数据总数：{len(trip)}")
print(f"筛选后的数据总数：{len(trip_clean)}")
print(f"剔除的数据总数：{len(trip) - len(trip_clean)}")


原始数据总数：669959
筛选后的数据总数：669663
剔除的数据总数：296


`.copy()` 的作用是**物理克隆**，它会在电脑内存里开辟一块全新的空间，把数据完完整整地复制过去，并彻底剪断与原始数据的任何联系。

In [13]:
# 处理zip_code列的缺失值，由于原数据太干净，在此复制原数据并注入不同的脏数据类型,模拟工程化清洗流程
import numpy as np
import random

# 将脏数据创建为新的一列并转为字符串类型
trip_clean['zip_code_dirty'] = trip_clean['zip_code'].astype(str)

# 打印新列的状态
print(f"当前总行数：{len(trip_clean)}")
print(trip_clean['zip_code_dirty'].value_counts().head())

# 随机抽取5%的行，将其内容修改为带有前后空格的自定义（94107）邮编
trip_clean.loc[trip_clean.sample(frac=0.05).index,'zip_code_dirty'] = '  94107  '
# 验证结果，正常邮编的长度为5，带了前后两个空格，长度变成9
print(trip_clean['zip_code_dirty'].str.len().value_counts()[9])

# 注入黑名单占位符（'?','-','N/A','Unknown','-999'）
placeholders = ['?','-','N/A','Unknown','-999']
trip_clean.loc[trip_clean.sample(frac=0.05).index,'zip_code_dirty'] = [
    random.choice(placeholders) for _ in range(int(len(trip_clean)*0.05))
]
# 验证结果
print(trip_clean['zip_code_dirty'].value_counts().reindex(placeholders))

# 注入高基数乱码
trip_clean.loc[trip_clean.sample(frac=0.05).index,'zip_code_dirty']= 'Error_ID_' + trip_clean.sample(frac=0.05).index.astype(str)
# 验证结果
print(trip_clean['zip_code_dirty'].nunique())

# 注入9位长邮编，带横杠
long_zip_index = trip_clean.sample(frac=0.05).index
trip_clean.loc[long_zip_index,'zip_code_dirty'] = '94107-1234'

# 注入带前缀的混合格式
mixed_zip_index = trip_clean.sample(frac=0.05).index
trip_clean.loc[mixed_zip_index,'zip_code_dirty'] = 'ZIP-941'

# 最终验证
print(f"最终验证：")
print(trip_clean['zip_code_dirty'].str.len().value_counts().sort_index())

当前总行数：669663
zip_code_dirty
94107    78695
94105    42668
94133    31353
94103    26668
94111    21407
Name: count, dtype: int64
33488
zip_code_dirty
?          6760
-          6623
N/A        6681
Unknown    6694
-999       6725
Name: count, dtype: int64
40682
最终验证：
zip_code_dirty
1      12152
2       1988
3      19520
4      11278
5     495419
6        236
7      39363
8         95
9      27288
10     32143
11         3
12        43
13       377
14      4113
15     25645
Name: count, dtype: int64


### 💡 核心知识点
* **`random.choice`**:随机抓取器。用于从列表中随机选出一个成员。
    * **实例**：
    ```python
    import random
    box = ['红包','谢谢惠顾','一等奖']
    result = random.choice(box)
    print(f"随机抓到了：{result}")
    # 每次运行结果可能都不一样
    ```
* **`reindex`**:强制对账单。用于按照指定索引重新排序并对其数据。
    * **演示示例**：
    ```python
    import pandas as pd
    # 现在的成绩单（只有张三和李四）
    scores = pd.Series([90,85]),index=['张三','李四'])

    # 老师手里的"全班点名册"(想要：张三、李四、王五)
    class_list = ['张三','李四','王五']

    # 强制按点名册对账
    final_report = scores.renidex(class_list)
    print(final_report)

    # 结果：王五没成绩，会自动补上NaN(空值)
    ```

### 接下来对`zip_code_dirty`做清洗练习：
* **清洗前的逻辑思考**：

* **类型对齐：**

 强制转换字段为 str 类型，确保清洗工具（.str）全量可用。

* **物理修复：**

 使用 replace(r'\s+', '', regex=True) 彻底挤压掉包括首位、末尾及中间的所有空白字符，最大限度抢救如 "94 107" 类的有效数据。

* **合规判定（白名单）：**

 应用正则 r'^\d{5}$'。

* **注意：**

原始数据中本就存在的 NaN 是不需要“转化”的，它们应该被保持原样，而不是被当成脏数据处理（虽然结果都是空，但逻辑出发点不同）。

* **Pass：**

符合“5位纯数字”模具的数据被精准保留。

* **Fail：**

所有不合规的乱码（如 ZIP-941）、占位符（如 ?）、逻辑空字符串（如 nan）统一重置为标准的 np.nan。

* **结果审计：**

通过 value_counts() 和 isna().sum() 验证清洗后的数据分布，确保最终只剩下 5 位合法邮编和标准空值。

In [19]:
# 强制转换trip_clean['zip_code_dirty']为'str'类型，确保清洗工具（.str）全量可用
trip_clean['zip_code_dirty'] = trip_clean['zip_code_dirty'].astype(str)

# 利用'replace'去掉首尾以及字符串中间的空格，保留因空格影响的有效数据
trip_clean['zip_code_dirty'] = trip_clean['zip_code_dirty'].str.replace(r'\s+','',regex=True)

# 定义白名单
pattern = r'^\d{5}$'

# 提取有效字段（包括原本就存在的NaN）
is_valid_or_nan = trip_clean['zip_code_dirty'].str.match(pattern,na=True)

# 覆盖不符合有效或NaN的字段
trip_clean['zip_code_dirty'] = trip_clean['zip_code_dirty'].where(is_valid_or_nan,np.nan)

# 看看现在长度不等于 5 的还有谁？
# 理论上，除了 NaN，结果应该是空的！
print(trip_clean[trip_clean['zip_code_dirty'].str.len() != 5]['zip_code_dirty'].unique())


[nan]


In [ ]:
# 进阶优化
print(f"优化前的内存占用：")
trip_clean.info()

print("\n" + "=" * 40 + "\n")

# 降级duration
trip_clean['duration'] = trip_clean['duration'].astype('int32')

# 查看站点名称列的内容
print(trip_clean[['start_station_name','end_station_name']].head())

# 转换站点名称为分类变量（category）
trip_clean['start_station_name'] = trip_clean['start_station_name'].astype('category')
trip_clean['end_station_name'] = trip_clean['end_station_name'].astype('category')

# 查看转换后站点名称列内容
print(trip_clean[['start_station_name','end_station_name']].head())

# 查看转换后的内存占用
print(trip_clean.info())

优化前的内存占用：
<class 'pandas.core.frame.DataFrame'>
Index: 669663 entries, 0 to 669958
Data columns (total 12 columns):
 #   Column              Non-Null Count   Dtype         
---  ------              --------------   -----         
 0   id                  669663 non-null  object        
 1   duration            669663 non-null  int64         
 2   start_date          669663 non-null  datetime64[ns]
 3   start_station_name  669663 non-null  object        
 4   start_station_id    669663 non-null  object        
 5   end_date            669663 non-null  datetime64[ns]
 6   end_station_name    669663 non-null  object        
 7   end_station_id      669663 non-null  object        
 8   bike_id             669663 non-null  object        
 9   subscription_type   669663 non-null  object        
 10  zip_code            669663 non-null  object        
 11  duration_hours      669663 non-null  float64       
dtypes: datetime64[ns](2), float64(1), int64(1), object(8)
memory usage: 66.4+ MB


  

: 

: 

## 为什么优化前后，打印出来的数据内容是一样的？
* **解析**
    * 因为运行`print`命令时，python会将分类编号映射的内容翻译给我们，实际上内部存储的不再是原始数据中的分类标签，而是分类编号，这样大大降低了内存占用

In [ ]:
# 检查每个id是否存在对应多个站点名称的错误映射

# 由于存在start_station_id和end_station_id两中id，因此在检查其错误映射之前，我们需要将两种id以及其对应的name，合并成一张表，便于统一处理

# 统一两种id与name的名称
starts = trip_clean[['start_station_id','start_station_name']].rename(
    columns={'start_station_id':'station_id','start_station_name':'station_name'}
)
ends = trip_clean[['end_station_id','end_station_name']].rename(
    columns={'end_station_id':'station_id','end_station_name':'station_name'}
)

# 合并两张表
all_stations = pd.concat([starts,ends],ignore_index=True)

# 检查每个id对应的站点名字数量
inconsistent_counts = all_stations.groupby('station_id')['station_name'].nunique()
print(inconsistent_counts)

# 找出名字数量大于1的id
inconsistent_ids = inconsistent_counts[inconsistent_counts >1]
print(inconsistent_ids)



station_id
10    1
11    1
12    1
13    1
14    1
     ..
80    2
82    1
83    1
84    1
9     1
Name: station_name, Length: 70, dtype: int64
station_id
25    2
46    2
47    2
80    2
Name: station_name, dtype: int64


: 

: 

In [ ]:
# 提取所有异常站点的ID
bad_ids = inconsistent_ids.index
print(bad_ids)

#  用 isin() 方法，在全局大表中把这些“嫌疑犯”的所有记录都捞出来
dirty_records = all_stations[all_stations['station_id'].isin(bad_ids)]
print(dirty_records)

# 去重！因为 80 号可能出现了几万次，我们只想看它到底有哪几种不同的名字组合
unique_dirty_records = dirty_records.drop_duplicates()
print(unique_dirty_records)

print(unique_dirty_records.sort_values('station_id'))

Index(['25', '46', '47', '80'], dtype='object', name='station_id')
        station_id           station_name
15              47        Post at Kearney
23              47        Post at Kearney
42              46  Washington at Kearney
64              47        Post at Kearney
127             47        Post at Kearney
...            ...                    ...
1339184         47         Post at Kearny
1339187         46   Washington at Kearny
1339188         46   Washington at Kearny
1339189         47         Post at Kearny
1339217         46   Washington at Kearny

[34457 rows x 2 columns]
       station_id                     station_name
15             47                  Post at Kearney
42             46            Washington at Kearney
1484           25                 Broadway at Main
101432         80       San Jose Government Center
143953         47                   Post at Kearny
143963         46             Washington at Kearny
144494         80  Santa Clara County Civic Ce

: 

: 